In [ ]:
# CELL 1: Import semua library
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile
import time
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

print("Library berhasil diimport")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# CELL 2: Setup Kaggle dan download dataset
import json

print("SETUP KAGGLE")

!mkdir -p ~/.kaggle

kaggle_json = {
    "username": "wildansani",
    "key": "fc1607894987c6602018c570db0106b0"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

!chmod 600 ~/.kaggle/kaggle.json

!pip install kaggle -q

print("Download dataset")
!kaggle datasets download techsash/waste-classification-data

print("Ekstrak dataset")
with zipfile.ZipFile('waste-classification-data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

os.remove('waste-classification-data.zip')

print("Dataset siap")

In [ ]:
# CELL 3: Cari path TRAIN dan TEST
data_dir = '/content/data'
train_path = None
test_path = None

for root, dirs, files in os.walk(data_dir):
    for dir_name in dirs:
        if dir_name == 'TRAIN':
            train_path = os.path.join(root, dir_name)
        if dir_name == 'TEST':
            test_path = os.path.join(root, dir_name)

print(f"TRAIN: {train_path}")
print(f"TEST: {test_path}")

for cls in os.listdir(train_path):
    train_count = len(os.listdir(os.path.join(train_path, cls)))
    test_count = len(os.listdir(os.path.join(test_path, cls)))
    print(f"{cls}: Train={train_count}, Test={test_count}")

In [ ]:
# CELL 4: Load dataset dengan augmentasi
print("LOAD DATASET")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"Training samples: {train_generator.samples}")
print(f"Testing samples: {test_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")

In [ ]:
# CELL 5: Build model dengan ResNet50
print("BUILD MODEL RESNET50")

tf.keras.backend.clear_session()

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

model.summary()

In [ ]:
# CELL 6: Training 15 epoch
print("TRAINING 15 EPOCH")

callbacks = [
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

start_time = time.time()

history = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

end_time = time.time()
print(f"Training time: {(end_time - start_time) / 60:.1f} minutes")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.2%}")

In [ ]:
# CELL 7: Fine-tuning 10 epoch
print("FINE-TUNING")

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

history_ft = model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

for key in history.history.keys():
    if key in history_ft.history:
        history.history[key].extend(history_ft.history[key])

print(f"Final val_accuracy: {max(history.history['val_accuracy']):.2%}")

In [ ]:
# CELL 9: Confusion Matrix
test_generator.reset()
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int)
true_classes = test_generator.classes

cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Organik', 'Anorganik'],
            yticklabels=['Organik', 'Anorganik'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(true_classes, predicted_classes,
                           target_names=['Organik', 'Anorganik']))

In [ ]:
# CELL 10: Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# CELL 11: Simpan dan download model
model.save('waste_classifier_final.h5')
print("Model saved as waste_classifier_final.h5")

from google.colab import files
files.download('waste_classifier_final.h5')

In [ ]:
# CELL 12: Test dengan upload gambar
from google.colab import files
from tensorflow.keras.preprocessing import image

print("Upload gambar untuk diuji")
uploaded = files.upload()

for filename in uploaded.keys():
    img = image.load_img(filename, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0

    pred = model.predict(img_array)[0][0]

    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.axis('off')

    if pred > 0.5:
        result = f"ANORGANIK\nConfidence: {pred:.2%}"
        color = 'green'
    else:
        result = f"ORGANIK\nConfidence: {1-pred:.2%}"
        color = 'orange'

    plt.title(result, color=color, fontsize=14, fontweight='bold')
    plt.show()

    print(f"Hasil: {result}")